In [63]:
# ********************** IMPORTS ********************** #

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast

# ********************** SETTINGS ********************** #

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [64]:
# ********************** FUNCTIONS ********************** #

# Parsing addres Col to Extract City

def address_parse(add_str):

    try:
        add_dict = json.loads(add_str.replace("'", "\""))
        return add_dict.get('city', None)
    
    except:
        return None
    
# Parsing crypto Col to Extract Coin Name

def crypto_parse(crypto_str):

    try:
        crypto_dict = json.loads(crypto_str.replace("'", "\""))
        return crypto_dict.get('coin', None)
    except:
        return None
    

# Parsing company Col to Extract Its Name and Location

def company_parse(company_str):

    try:
        company_dict = ast.literal_eval(company_str) if isinstance(company_str, str) else company_str

        name = company_dict.get('name', None)

        city = company_dict.get('address', {}).get('city', None)

        return name, city
    except:
        return None, None

# Flatten Dict Col

import ast
import pandas as pd

def flatten_dict_column(df, col):

    try:
        def convert(x):

            try:
                if isinstance(x, str):
                    return ast.literal_eval(x)
                return x
            
            except Exception:
                return None

        df[col] = df[col].apply(convert)

        new_cols = pd.json_normalize(df[col])

        df = pd.concat([df.drop(columns=col), new_cols], axis=1)

        return df

    except KeyError:
        print(f"Column '{col}' does not exist in the DataFrame.")
        return df

    except Exception as e:
        print(f"Error while flattening column '{col}': {e}")
        return df

# Parsing userAgent Col to Extract OS Type

def extract_os(df, userAgent):

    df[userAgent] = df[userAgent].str.extract(r'\(([^;]+)')
    return df


In [65]:
df = pd.read_csv('users.csv')

In [66]:
# ********************** APPLYING ADDRESS PARSE ********************** #

df['city'] = df['address'].apply(address_parse)
df.drop(columns='address', inplace=True)

# ********************** APPLYING CRYPTO PARSE ********************** #

df['crypto'] = df['crypto'].apply(crypto_parse) 

# ********************** APPLYING COMPANY PARSE ********************** #

df[['company_name', 'company_location']] = df['company'].apply(company_parse).apply(pd.Series)
df.drop(columns='company', inplace=True)

# ********************** FLATTEN BANK, HAIR COLS ********************** #

df = flatten_dict_column(df, 'bank')

df = flatten_dict_column(df, 'hair')
df.rename(columns={'color': 'hair_color'}, inplace=True)
df.rename(columns={'type': 'hair_type'}, inplace=True)

# ********************** EXTRACTING OS NAME ********************** #

df = extract_os(df, 'userAgent')



In [67]:
df.head()

,id,firstName,lastName,maidenName,age,gender,email,phone,username,password,birthDate,image,bloodGroup,height,weight,eyeColor,ip,macAddress,university,ein,ssn,userAgent,crypto,role,city,company_name,company_location,cardExpire,cardNumber,cardType,currency,iban,hair_color,hair_type
0,1,Emily,Johnson,Smith,29,female,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,emilyspass,1996-5-30,https://dummyjson.com/icon/emilys/128,O-,193.24,63.16,Green,42.48.100.32,47:fa:41:18:ec:eb,University of Wisconsin--Madison,977-175,900-590-289,Macintosh,Bitcoin,admin,Phoenix,"Dooley, Kozey and Cronin",San Francisco,05/28,3693233511855044,Diners Club International,GBP,GB74MH2UZLR9TRPHYNU8F8,Brown,Curly
1,2,Michael,Williams,NaN,36,male,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,michaelwpass,1989-8-10,https://dummyjson.com/icon/michaelw/128,B+,186.22,76.32,Red,12.13.116.142,79:15:78:99:60:aa,Ohio State University,912-602,108-953-962,Windows NT 10.0,Bitcoin,admin,Houston,Spinka - Dickinson,Los Angeles,01/30,3530633803003665,JCB,USD,DE26362283149158045865,Green,Straight
2,3,Sophia,Brown,NaN,43,female,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,sophiabpass,1982-11-6,https://dummyjson.com/icon/sophiab/128,O-,177.72,52.60,Hazel,214.225.51.195,12:a3:d3:6f:5c:5b,Pepperdine University,963-113,638-461-822,Windows NT 10.0,Bitcoin,admin,Washington,Schiller - Zieme,Dallas,10/27,6011212053392887,Discover,EUR,DE12191213468288004835,White,Wavy
3,4,James,Davis,NaN,46,male,james.davis@x.dummyjson.com,+49 614-958-9364,jamesd,jamesdpass,1979-5-4,https://dummyjson.com/icon/jamesd/128,AB+,193.31,62.10,Amber,101.118.131.66,10:7d:df:1f:97:58,University of Southern California,904-810,116-951-314,Macintosh,Bitcoin,admin,Seattle,Pagac and Sons,Fort Worth,07/30,5303440212268149,Mastercard,CAD,DE01300746880579852937,Blonde,Straight
4,5,Emma,Miller,Johnson,31,female,emma.miller@x.dummyjson.com,+91 759-776-1614,emmaj,emmajpass,1994-6-13,https://dummyjson.com/icon/emmaj/128,AB-,192.80,63.62,Green,224.126.22.183,32:b9:7e:8d:f5:e8,Northeastern University,403-505,526-210-885,Windows NT 10.0,Bitcoin,admin,Jacksonville,Graham - Gulgowski,San Antonio,07/30,5237188057591130,Mastercard,NZD,DE19182355652037133559,White,Straight


In [68]:
df.isna().sum()

id                    0
firstName             0
lastName              0
maidenName          148
age                   0
gender                0
email                 0
phone                 0
username              0
password              0
birthDate             0
image                 0
bloodGroup            0
height                0
weight                0
eyeColor              0
ip                    0
macAddress            0
university            0
ein                   0
ssn                   0
userAgent             0
crypto                0
role                  0
city                  0
company_name          0
company_location      0
cardExpire            0
cardNumber            0
cardType              0
currency              0
iban                  0
hair_color            0
hair_type             0
dtype: int64

In [69]:
df['maidenName'].fillna('Unknown')

0        Smith
1      Unknown
2      Unknown
3      Unknown
4      Johnson
        ...   
203    Unknown
204       Gray
205    Unknown
206    Unknown
207      Lopez
Name: maidenName, Length: 208, dtype: object

In [70]:
df.to_csv('cleaned_users.csv', index=False)